# LC8 — CIM-XML: from grid snapshots to time series (self-paced, ~40 min)

The Svedala CSVs you have trusted for two courses were *derived* — this is the source: four CIM-XML files in the CGMES standard, the format European TSOs actually exchange grids in. EG2130 introduced the structural side; here we recap it briefly and then focus on what EG2130 never used: **SSH files as operating-state snapshots, and a folder of them as a time series**. Material from this notebook appears in **Quiz 3**; Lab 6 does the real work.

## 1. Recap in one cell: four files, four jobs

| Profile | Answers | Changes |
|---|---|---|
| **EQ** (equipment) | what exists — buses, lines, machines, their parameters | rarely |
| **TP** (topology) | how it is connected right now | on switching |
| **SSH** (steady state hypothesis) | what everything is *doing* — load P/Q, dispatch | **every instant** |
| **SV** (state variables) | the solved result — voltages, flows | every instant |

Same grid, separated by *rate of change*. The provenance chain to your CSVs: ENTSO-E CGMES test model → these four files → a converter → `buses.csv` and friends. And the punchline of this session: EQ+TP is **the network**; a directory of timestamped SSH files is **its operational history**.

In [ ]:
# Install exactly what this notebook uses (re, zipfile, pathlib are stdlib).
%pip install pandas matplotlib --quiet

In [ ]:
from pathlib import Path
# Guard: this notebook expects to run from the notebooks/ folder of a clone of
# the course repository — the datasets live one level up in ../data/.
# Failing here, early and clearly, beats a confusing FileNotFoundError later.
assert Path("../data").exists(), (
    "Course data folder not found. Clone KTH-EG2140/course-material and open "
    "this notebook from its notebooks/ folder.")

In [ ]:
from pathlib import Path
CIM = Path("../data/svedala-cim")
for f in sorted(CIM.glob("*.xml")):
    print(f"{f.name:20s} {f.stat().st_size/1024:7.0f} kB")

## 2. Open one — it is only XML

Look at the raw text before any tool touches it. An SSH file is a list of statements: *this load draws 75 MW and 30 Mvar right now*:

In [ ]:
ssh_text = (CIM / "network_SSH.xml").read_text(encoding="utf-8")
print(ssh_text[:1200])

Everything hangs on the `rdf:about="#_8a28..."` identifiers — **mRIDs**, the machine-readable names. The SSH never says "AT111_T1_LAST"; human names live in EQ. Joining the two is Lab 6's first task, and it is ordinary data engineering:

In [ ]:
import re
# EQ: mRID -> name, for the loads
eq_text = (CIM / "network_EQ.xml").read_text(encoding="utf-8")
name_of = dict(re.findall(
    r'<cim:ConformLoad rdf:ID="(_[0-9a-f-]+)">.*?<cim:IdentifiedObject.name>([^<]+)<',
    eq_text, re.S))
# SSH: mRID -> P
p_of = {m: float(p) for m, p in re.findall(
    r'<cim:ConformLoad rdf:about="#(_[0-9a-f-]+)">\s*<cim:EnergyConsumer.p>([-\d.]+)', ssh_text)}
joined = {name_of[m]: p for m, p in p_of.items() if m in name_of}
print(len(joined), "loads joined; total", round(sum(joined.values())), "MW")
list(joined.items())[:5]

(Real tools use proper RDF parsers, not regex — but seeing the join once by hand removes the mystery. Also: the total you just computed should look familiar.)

## 3. A folder of snapshots IS a time series

`data/svedala-year/ssh_week.zip` holds 168 SSH files — one January week, one snapshot per hour, timestamps in the filenames and in `Model.scenarioTime`. Extract one day and watch a time series appear out of "static" network files:

In [ ]:
import zipfile
import pandas as pd
zf = zipfile.ZipFile("../data/svedala-year/ssh_week.zip")
day = sorted(n for n in zf.namelist() if "2025-01-15" in n)
print(len(day), "snapshots for 2025-01-15")
# Loop the 24 files of one day: read each snapshot, take its scenarioTime and
# the sum of all ConformLoad P values -> one (timestamp, total MW) pair each.
rows = {}
for n in day:
    txt = zf.read(n).decode("utf-8")
    ts = re.search(r"scenarioTime>([^<]+)<", txt).group(1)
    total = sum(float(p) for _, p in re.findall(
        r'<cim:ConformLoad rdf:about="#(_[0-9a-f-]+)">\s*<cim:EnergyConsumer.p>([-\d.]+)', txt))
    rows[pd.Timestamp(ts)] = total
series = pd.Series(rows).sort_index()
ax = series.plot(figsize=(9, 3), marker="o", title="Total load from 24 SSH snapshots — a time series")
ax.set_ylabel("MW");

One day, per hour, assembled from files that each describe a single instant. Lab 6 does this properly: all 168 hours, **per zone** (the EQ join gives each load its zone via the CSVs), into a tidy DataFrame checked against the course dataset.

## 4. Shapes of time series — the vocabulary

The same physical reality arrives in different shapes, and the shape decides what you can model:

- **Market data vs SCADA**: ENTSO-E gives *zonal, hourly, settled* numbers; a control room gets *per-element, per-second, raw* measurements. Different questions, different data.
- **Per-element vs per-zone**: the SSH gives every load; Lab 6 aggregates to zones — an irreversible choice. Aggregate late.
- **Wide vs long**: one column per zone (wide — good for modelling) or one row per (time, zone, value) (long — good for storage and joins). `pd.melt` / `pivot` convert; know both.

In [ ]:
wide = pd.DataFrame({"ZON_A": [1.0, 2.0], "ZON_B": [3.0, 4.0]},
                    index=pd.to_datetime(["2025-01-01 00:00", "2025-01-01 01:00"]))
long = wide.reset_index(names="timestamp").melt(id_vars="timestamp",
                                                var_name="zone", value_name="mw")
print(wide, "\n\n", long)

## Self-check

In [ ]:
assert len(joined) >= 55, "EQ/SSH join lost too many loads"
assert abs(sum(joined.values()) - 10981) < 200, "SSH base-case total looks wrong"
assert len(series) == 24 and series.max() > series.min() * 1.1
print("ALL OK — you have read CIM by hand. Lab 6 industrialises it.")